# Same-Game Model Comparison

This notebook compares balanced logistic regression and XGBoost using same-game box score style features from `Tommy_Award_Player_Game_Table.csv`.

This is a postgame/descriptive setup: the model uses what players did in the game itself to approximate who won the Tommy Award.

In [9]:
# Import the libraries used for data handling, preprocessing, evaluation,
# and the two model families compared in this notebook.
import importlib
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Basic configuration shared across the notebook.
INPUT_PATH = "Tommy_Award_Player_Game_Table.csv"
RANDOM_STATE = 42
MIN_TRAIN_SEASONS = 3

In [10]:
# Helper functions for loading the dataset, choosing the same-game feature set,
# creating season-based train/test splits, and building numeric preprocessing.
def load_dataset(path: str = INPUT_PATH) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"GAME_ID": str}).copy()
    df["game_date"] = pd.to_datetime(df["game_date"], format="mixed")

    # Keep only players who actually played in the game.
    df = df[df["minutes_decimal"] > 0].copy()

    if "season" not in df.columns:
        start_year = df["game_date"].dt.year.where(df["game_date"].dt.month >= 10, df["game_date"].dt.year - 1)
        df["season"] = start_year.astype(str) + "-" + (start_year + 1).astype(str).str[-2:]

    return df


def sorted_seasons(seasons: list[str]) -> list[str]:
    return sorted(seasons, key=lambda season: int(str(season).split("-")[0]))


def get_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_cols = [
        "minutes_decimal",
        "points",
        "reboundsOffensive",
        "reboundsDefensive",
        "reboundsTotal",
        "assists",
        "steals",
        "blocks",
        "turnovers",
        "foulsPersonal",
        "plusMinusPoints",
        "fieldGoalsMade",
        "fieldGoalsAttempted",
        "threePointersMade",
        "threePointersAttempted",
        "freeThrowsMade",
        "stocks",
        "points_per_min",
        "oreb_per_min",
        "reb_per_min",
        "ast_per_min",
        "hustle_proxy",
        "points_rank",
        "reboundsOffensive_rank",
        "reboundsTotal_rank",
        "assists_rank",
        "steals_rank",
        "blocks_rank",
        "plusMinusPoints_rank",
        "minutes_decimal_rank",
    ]
    return numeric_cols, []


def get_model_feature_columns(
    model_name: str,
    numeric_cols: list[str],
    categorical_cols: list[str],
) -> tuple[list[str], list[str]]:
    return numeric_cols.copy(), categorical_cols.copy()


def walk_forward_season_splits(
    df: pd.DataFrame,
    min_train_seasons: int = MIN_TRAIN_SEASONS,
) -> list[tuple[list[str], str]]:
    seasons = sorted_seasons(df["season"].dropna().unique().tolist())
    splits = []
    for idx in range(min_train_seasons, len(seasons)):
        splits.append((seasons[:idx], seasons[idx]))
    return splits


def latest_season_holdout_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    seasons = sorted_seasons(df["season"].dropna().unique().tolist())
    train_df = df[df["season"].isin(seasons[:-1])].copy()
    test_df = df[df["season"] == seasons[-1]].copy()
    return train_df, test_df


def build_preprocessor(numeric_cols: list[str], categorical_cols: list[str]) -> ColumnTransformer:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
        ]
    )

In [34]:
# Model-building and evaluation helpers.
# This block defines how each model is trained and how predictions are scored.
def build_logistic_pipeline(numeric_cols: list[str], categorical_cols: list[str]) -> Pipeline:
    return Pipeline(
        steps=[
            ("prep", build_preprocessor(numeric_cols, categorical_cols)),
            (
                "clf",
                LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def build_xgboost_pipeline(
    numeric_cols: list[str],
    categorical_cols: list[str],
    scale_pos_weight: float,
) -> Pipeline:
    XGBClassifier = importlib.import_module("xgboost").XGBClassifier
    return Pipeline(
        steps=[
            ("prep", build_preprocessor(numeric_cols, categorical_cols)),
            (
                "clf",
                XGBClassifier(
                    # Binary classification because each row is winner vs non-winner.
                    objective="binary:logistic",
                    # Log loss matches our probability outputs and punishes overconfident mistakes.
                    eval_metric="logloss",
                    # More trees, but each one learns more slowly.
                    n_estimators=400,
                    # Shallower trees reduce overfitting to weird one-game stat lines.
                    max_depth=3,
                    # Smaller learning rate makes the model update more carefully.
                    learning_rate=0.03,
                    # Use only part of the rows per tree to improve generalization.
                    subsample=0.80,
                    # Use only part of the features per tree so the model does not rely too heavily on one stat.
                    colsample_bytree=0.70,
                    # Require a little more evidence before creating detailed splits.
                    min_child_weight=3,
                    # Ignore tiny improvements that often just fit noise.
                    gamma=0.10,
                    # L1 regularization: encourages simpler trees.
                    reg_alpha=0.30,
                    # L2 regularization: shrinks overly aggressive fits.
                    reg_lambda=2.00,
                    random_state=RANDOM_STATE,
                    n_jobs=4,
                    # Upweight the rare winner rows so the model pays attention to them.
                    scale_pos_weight=scale_pos_weight,
                ),
            ),
        ]
    )


def available_model_names() -> tuple[list[str], str | None]:
    model_names = ["balanced_logistic"]
    xgboost_error = None
    try:
        importlib.import_module("xgboost")
        model_names.append("xgboost")
    except Exception as exc:
        xgboost_error = str(exc).strip()
    return model_names, xgboost_error


def score_predictions(scored_df: pd.DataFrame) -> dict[str, float]:
    pred_winners = (
        scored_df.sort_values(["GAME_ID", "pred_prob"], ascending=[True, False])
        .groupby("GAME_ID")
        .head(1)
        .copy()
    )
    top3 = (
        scored_df.sort_values(["GAME_ID", "pred_prob"], ascending=[True, False])
        .groupby("GAME_ID")
        .head(3)
        .copy()
    )

    y_true = scored_df["y"]
    y_pred = (scored_df["pred_prob"] >= 0.5).astype(int)

    return {
        "game_level_accuracy": pred_winners["y"].mean(),
        "top3_accuracy": top3.groupby("GAME_ID")["y"].max().mean(),
        "row_accuracy": (y_pred == y_true).mean(),
        "row_logloss": log_loss(y_true, scored_df["pred_prob"], labels=[0, 1]),
        "row_auc": roc_auc_score(y_true, scored_df["pred_prob"]),
    }


def fit_and_score_model(
    model_name: str,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    numeric_cols: list[str],
    categorical_cols: list[str],
) -> tuple[Pipeline, pd.DataFrame, dict[str, float]]:
    model_numeric_cols, model_categorical_cols = get_model_feature_columns(
        model_name,
        numeric_cols,
        categorical_cols,
    )
    X_train = train_df[model_numeric_cols + model_categorical_cols]
    X_test = test_df[model_numeric_cols + model_categorical_cols]
    y_train = train_df["y"]

    # XGBoost uses this ratio to account for the class imbalance.
    pos = max(int(y_train.sum()), 1)
    neg = max(int((y_train == 0).sum()), 1)
    scale_pos_weight = neg / pos

    if model_name == "balanced_logistic":
        model = build_logistic_pipeline(model_numeric_cols, model_categorical_cols)
    elif model_name == "xgboost":
        model = build_xgboost_pipeline(model_numeric_cols, model_categorical_cols, scale_pos_weight)
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    model.fit(X_train, y_train)

    scored_df = test_df.copy()
    scored_df["pred_prob"] = model.predict_proba(X_test)[:, 1]
    metrics = score_predictions(scored_df)
    return model, scored_df, metrics

In [40]:
# Load the same-game dataset, identify the chosen feature set,
# and print a small summary of the modeling setup.
df = load_dataset()
numeric_cols, categorical_cols = get_feature_columns(df)
model_names, xgboost_error = available_model_names()
train_df, latest_test_df = latest_season_holdout_split(df)
latest_season = sorted_seasons(df["season"].dropna().unique().tolist())[-1]
feature_count_summary = ", ".join(
    f"{model_name}={len(get_model_feature_columns(model_name, numeric_cols, categorical_cols)[0])}"
    for model_name in model_names
)

summary_df = pd.DataFrame(
    {
        "item": [
            "rows",
            "numeric_features",
            "categorical_features",
            "train_games",
            "test_games",
            "train_positive_rate",
            "test_season",
            "available_models",
        ],
        "value": [
            len(df),
            feature_count_summary,
            len(categorical_cols),
            train_df["GAME_ID"].nunique(),
            latest_test_df["GAME_ID"].nunique(),
            round(train_df["y"].mean(), 4),
            latest_season,
            ", ".join(model_names),
        ],
    }
)

display(summary_df)

if xgboost_error is not None:
    print("XGBoost is configured but unavailable in this environment.")
    print(xgboost_error)

,item,value
0,rows,8377
1,numeric_features,30
2,categorical_features,0
3,train_games,707
4,test_games,71
5,train_positive_rate,0.0929
6,test_season,2025-26
7,available_models,"balanced_logistic, xgboost"


In [46]:
# Walk-forward evaluation tests each future season using only earlier seasons for training.
# This is the most realistic way to check how the model generalizes over time.
walk_forward_rows = []

for train_seasons, test_season in walk_forward_season_splits(df):
    split_train_df = df[df["season"].isin(train_seasons)].copy()
    split_test_df = df[df["season"] == test_season].copy()

    for model_name in model_names:
        _, _, metrics = fit_and_score_model(
            model_name,
            split_train_df,
            split_test_df,
            numeric_cols,
            categorical_cols,
        )
        walk_forward_rows.append(
            {
                "model": model_name,
                "test_season": test_season,
                **metrics,
            }
        )

walk_forward_df = pd.DataFrame(walk_forward_rows)
walk_forward_summary_df = (
    walk_forward_df.groupby("model", as_index=False)[
        ["game_level_accuracy", "top3_accuracy", "row_accuracy", "row_logloss", "row_auc"]
    ]
    .mean()
    .round(4)
)

display(walk_forward_df.round(4))
display(walk_forward_summary_df)

,model,test_season,game_level_accuracy,top3_accuracy,row_accuracy,row_logloss,row_auc
0,balanced_logistic,2019-20,0.4861,0.7917,0.7985,0.4208,0.8865
1,xgboost,2019-20,0.4861,0.7500,0.8195,0.3449,0.8783
2,balanced_logistic,2020-21,0.2222,0.7083,0.7754,0.5129,0.8134
3,xgboost,2020-21,0.1944,0.6389,0.7842,0.4258,0.7901
4,balanced_logistic,2021-22,0.3625,0.7625,0.7587,0.4973,0.8455
5,xgboost,2021-22,0.3750,0.7500,0.7933,0.3642,0.8605
6,balanced_logistic,2022-23,0.2317,0.6707,0.7118,0.5697,0.7932
7,xgboost,2022-23,0.2683,0.7805,0.7541,0.4482,0.8062
8,balanced_logistic,2023-24,0.3544,0.7089,0.6942,0.5722,0.8289
9,xgboost,2023-24,0.3671,0.7848,0.7172,0.4656,0.8358


,model,game_level_accuracy,top3_accuracy,row_accuracy,row_logloss,row_auc
0,balanced_logistic,0.3358,0.7370,0.7434,0.5165,0.8375
1,xgboost,0.3511,0.7395,0.7617,0.4197,0.8359


In [44]:
xgb_numeric_cols, _ = get_model_feature_columns("xgboost", numeric_cols, categorical_cols)
log_numeric_cols, _ = get_model_feature_columns("balanced_logistic", numeric_cols, categorical_cols)

xgb_model = trained_models["xgboost"].named_steps["clf"]

importance_df = pd.DataFrame({
    "feature": xgb_numeric_cols,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df.head(20))

log_model = trained_models["balanced_logistic"].named_steps["clf"]

coef_df = pd.DataFrame({
    "feature": log_numeric_cols,
    "coefficient": log_model.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

display(coef_df.head(20))

,feature,importance
11,fieldGoalsMade,0.275125
1,points,0.118289
21,hustle_proxy,0.056471
22,points_rank,0.042681
17,points_per_min,0.034029
28,plusMinusPoints_rank,0.032098
6,steals,0.028314
2,reboundsOffensive,0.023263
7,blocks,0.023072
4,reboundsTotal,0.022756


,feature,coefficient,abs_coefficient
12,fieldGoalsAttempted,-1.637827,1.637827
22,points_rank,-1.209447,1.209447
11,fieldGoalsMade,0.860727,0.860727
1,points,0.542882,0.542882
0,minutes_decimal,0.487992,0.487992
19,reb_per_min,0.460364,0.460364
28,plusMinusPoints_rank,-0.429218,0.429218
17,points_per_min,0.416811,0.416811
5,assists,0.323337,0.323337
14,threePointersAttempted,0.322241,0.322241


In [42]:
# Train each model on all earlier seasons and evaluate on the newest season only.
# This gives a single final holdout result you can report as the main test score.
latest_results = []
scored_outputs = {}
trained_models = {}

for model_name in model_names:
    model, scored_df, metrics = fit_and_score_model(
        model_name,
        train_df,
        latest_test_df,
        numeric_cols,
        categorical_cols,
    )
    trained_models[model_name] = model
    scored_outputs[model_name] = scored_df
    latest_results.append({"model": model_name, **metrics})

latest_results_df = pd.DataFrame(latest_results).round(4)
display(latest_results_df)

,model,game_level_accuracy,top3_accuracy,row_accuracy,row_logloss,row_auc
0,balanced_logistic,0.3521,0.7606,0.7451,0.4953,0.8556
1,xgboost,0.3521,0.7042,0.7438,0.4394,0.8381


In [47]:
# Show five random games for each model so you can inspect how the rankings look.
# If the scored outputs are not already in memory, rebuild them first.
show_cols = [
    "GAME_ID",
    "game_date",
    "player_name",
    "winner_names",
    "y",
    "pred_prob",
    "minutes_decimal",
    "fieldGoalsMade",
    "points",
    "reboundsOffensive",
    "reboundsTotal",
    "assists",
    "steals",
    "blocks",
    "plusMinusPoints",
]

if "scored_outputs" not in globals():
    latest_results = []
    scored_outputs = {}
    trained_models = {}

    for model_name in model_names:
        model, scored_df, metrics = fit_and_score_model(
            model_name,
            train_df,
            latest_test_df,
            numeric_cols,
            categorical_cols,
        )
        trained_models[model_name] = model
        scored_outputs[model_name] = scored_df
        latest_results.append({"model": model_name, **metrics})

    latest_results_df = pd.DataFrame(latest_results).round(4)
    display(latest_results_df)

RANDOM_GAME_SEED = 42

for model_name, scored_df in scored_outputs.items():
    print(f"\n=== Random games: {model_name} ===")
    random_game_ids = (
        scored_df["GAME_ID"]
        .drop_duplicates()
        .sample(n=min(5, scored_df["GAME_ID"].nunique()), random_state=RANDOM_GAME_SEED)
        .tolist()
    )

    for game_id in random_game_ids:
        game_rows = (
            scored_df[scored_df["GAME_ID"] == game_id]
            .sort_values("pred_prob", ascending=False)[show_cols]
        )
        print(f"\nGAME_ID: {game_id}")
        display(game_rows)


=== Random games: balanced_logistic ===

GAME_ID: 0022500334


,GAME_ID,game_date,player_name,winner_names,y,pred_prob,minutes_decimal,fieldGoalsMade,points,reboundsOffensive,reboundsTotal,assists,steals,blocks,plusMinusPoints
7844,0022500334,2025-12-04,Jordan Walsh,Jordan Walsh,1,0.932336,29.683333,8,22,2,7,3,2,1,20.0
7847,0022500334,2025-12-04,Derrick White,Jordan Walsh,0,0.870904,28.766667,12,30,0,7,9,1,0,17.0
7846,0022500334,2025-12-04,Neemias Queta,Jordan Walsh,0,0.824170,21.050000,7,17,2,4,2,1,4,29.0
7848,0022500334,2025-12-04,Payton Pritchard,Jordan Walsh,0,0.588218,32.150000,8,20,5,6,8,1,0,27.0
7850,0022500334,2025-12-04,Hugo González,Jordan Walsh,0,0.572174,17.666667,6,14,3,5,0,0,1,23.0
7849,0022500334,2025-12-04,Anfernee Simons,Jordan Walsh,0,0.168377,25.450000,4,16,0,2,3,2,0,34.0
7852,0022500334,2025-12-04,Baylor Scheierman,Jordan Walsh,0,0.134908,25.016667,2,5,1,6,3,3,0,42.0
7851,0022500334,2025-12-04,Josh Minott,Jordan Walsh,0,0.126827,22.033333,4,11,1,4,2,1,0,2.0
7853,0022500334,2025-12-04,Luka Garza,Jordan Walsh,0,0.080199,8.233333,1,5,1,2,1,1,1,9.0
7845,0022500334,2025-12-04,Sam Hauser,Jordan Walsh,0,0.045437,20.516667,2,6,0,2,3,1,0,4.0



GAME_ID: 0022500017


,GAME_ID,game_date,player_name,winner_names,y,pred_prob,minutes_decimal,fieldGoalsMade,points,reboundsOffensive,reboundsTotal,assists,steals,blocks,plusMinusPoints
7602,0022500017,2026-01-19,Payton Pritchard,Jaylen Brown,0,0.752859,33.650000,5,17,1,3,3,0,0,2.0
7599,0022500017,2026-01-19,Sam Hauser,Jaylen Brown,0,0.745328,27.350000,6,16,0,5,1,0,0,2.0
7600,0022500017,2026-01-19,Neemias Queta,Jaylen Brown,0,0.606330,28.333333,1,10,1,8,1,3,1,15.0
7598,0022500017,2026-01-19,Jaylen Brown,Jaylen Brown,1,0.561834,38.916667,11,32,4,11,2,0,1,2.0
7603,0022500017,2026-01-19,Luka Garza,Jaylen Brown,0,0.325600,15.200000,3,10,2,4,0,2,0,-8.0
7601,0022500017,2026-01-19,Derrick White,Jaylen Brown,0,0.216710,37.733333,1,4,5,9,5,0,2,9.0
7604,0022500017,2026-01-19,Anfernee Simons,Jaylen Brown,0,0.076077,24.683333,4,9,1,3,0,1,0,-13.0
7608,0022500017,2026-01-19,Baylor Scheierman,Jaylen Brown,0,0.051613,14.666667,1,3,0,1,1,1,0,-3.0
7606,0022500017,2026-01-19,Jordan Walsh,Jaylen Brown,0,0.020605,16.083333,1,2,0,1,0,0,0,-7.0
7605,0022500017,2026-01-19,Hugo González,Jaylen Brown,0,0.011781,1.783333,0,0,0,0,0,0,0,-2.0



GAME_ID: 0022500741


...

,GAME_ID,game_date,player_name,winner_names,y,pred_prob,minutes_decimal,fieldGoalsMade,points,reboundsOffensive,reboundsTotal,assists,steals,blocks,plusMinusPoints
8133,0022500741,2026-02-06,Nikola Vučević,Payton Pritchard,0,0.899191,27.883333,4,11,6,12,4,2,0,11.0
8132,0022500741,2026-02-06,Payton Pritchard,Payton Pritchard,1,0.631453,35.633333,8,24,0,1,1,1,0,4.0
8130,0022500741,2026-02-06,Jaylen Brown,Payton Pritchard,0,0.604555,33.083333,11,29,1,7,2,0,1,-16.0
8131,0022500741,2026-02-06,Derrick White,Payton Pritchard,0,0.461659,41.266667,6,21,1,3,5,1,4,6.0
8135,0022500741,2026-02-06,Baylor Scheierman,Payton Pritchard,0,0.222120,18.633333,2,5,0,7,2,1,1,17.0
8129,0022500741,2026-02-06,Neemias Queta,Payton Pritchard,0,0.030166,21.833333,1,2,4,11,1,0,0,-6.0
8127,0022500741,2026-02-06,Sam Hauser,Payton Pritchard,0,0.029510,35.300000,1,2,1,3,2,1,1,3.0
8134,0022500741,2026-02-06,Hugo González,Payton Pritchard,0,0.013109,16.216667,1,2,0,3,0,1,0,-2.0
8128,0022500741,2026-02-06,Luka Garza,Payton Pritchard,0,0.005209,10.150000,1,2,0,3,0,0,0,-7.0



GAME_ID: 0022500049


,GAME_ID,game_date,player_name,winner_names,y,pred_prob,minutes_decimal,fieldGoalsMade,points,reboundsOffensive,reboundsTotal,assists,steals,blocks,plusMinusPoints
7644,0022500049,2025-11-21,Neemias Queta,Neemias Queta,1,0.855933,29.183333,8,16,5,12,3,0,1,-5.0
7642,0022500049,2025-11-21,Jaylen Brown,Neemias Queta,0,0.714160,31.650000,9,26,1,8,4,2,0,-4.0
7647,0022500049,2025-11-21,Anfernee Simons,Neemias Queta,0,0.709688,33.033333,10,23,0,3,4,2,0,-10.0
7643,0022500049,2025-11-21,Jordan Walsh,Neemias Queta,0,0.623750,24.133333,4,9,2,6,0,1,0,8.0
7645,0022500049,2025-11-21,Payton Pritchard,Neemias Queta,0,0.173443,34.750000,5,13,1,3,4,0,0,-13.0
7649,0022500049,2025-11-21,Luka Garza,Neemias Queta,0,0.102331,16.833333,2,7,3,4,1,0,0,-5.0
7652,0022500049,2025-11-21,Hugo González,Neemias Queta,0,0.015181,5.216667,0,0,0,0,0,1,0,12.0
7646,0022500049,2025-11-21,Derrick White,Neemias Queta,0,0.012571,32.583333,2,6,0,1,4,1,0,-7.0
7651,0022500049,2025-11-21,Baylor Scheierman,Neemias Queta,0,0.012525,11.166667,2,5,0,2,1,0,0,-8.0
7648,0022500049,2025-11-21,Sam Hauser,Neemias Queta,0,0.004608,14.200000,0,0,1,4,2,0,0,-6.0



GAME_ID: 0022500847


,GAME_ID,game_date,player_name,winner_names,y,pred_prob,minutes_decimal,fieldGoalsMade,points,reboundsOffensive,reboundsTotal,assists,steals,blocks,plusMinusPoints
8197,0022500847,2026-02-25,Derrick White,Derrick White,1,0.674851,32.666667,6,20,3,6,3,1,3,-20.0
8193,0022500847,2026-02-25,Jaylen Brown,Derrick White,0,0.402647,33.766667,7,23,0,11,3,0,3,-12.0
8195,0022500847,2026-02-25,Neemias Queta,Derrick White,0,0.172008,28.350000,4,10,0,4,1,0,1,-7.0
8196,0022500847,2026-02-25,Baylor Scheierman,Derrick White,0,0.080267,24.533333,3,9,1,4,0,0,0,-16.0
8202,0022500847,2026-02-25,Jordan Walsh,Derrick White,0,0.071326,13.683333,2,4,0,3,0,0,0,-4.0
8194,0022500847,2026-02-25,Sam Hauser,Derrick White,0,0.046032,24.366667,2,5,2,4,1,0,0,-2.0
8200,0022500847,2026-02-25,Hugo González,Derrick White,0,0.033064,10.166667,2,5,0,2,1,1,0,-4.0
8203,0022500847,2026-02-25,Luka Garza,Derrick White,0,0.011379,5.550000,0,0,2,2,1,0,0,-2.0
8201,0022500847,2026-02-25,Ron Harper Jr.,Derrick White,0,0.007279,10.650000,1,3,0,1,2,0,0,-4.0
8198,0022500847,2026-02-25,Nikola Vučević,Derrick White,0,0.006647,22.966667,1,2,1,8,1,0,0,-9.0


In [39]:
# Optional reference table showing exactly which same-game columns are being used as model inputs.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

print("Balanced logistic numeric features")
display(
    pd.DataFrame(
        {"numeric_feature": get_model_feature_columns("balanced_logistic", numeric_cols, categorical_cols)[0]}
    )
)

print("XGBoost numeric features")
display(
    pd.DataFrame(
        {"numeric_feature": get_model_feature_columns("xgboost", numeric_cols, categorical_cols)[0]}
    )
)

print("Categorical features")
display(pd.DataFrame({"categorical_feature": categorical_cols}))

Numeric features


,numeric_feature
0,minutes_decimal
1,points
2,reboundsOffensive
3,reboundsDefensive
4,reboundsTotal
5,assists
6,steals
7,blocks
8,turnovers
9,foulsPersonal


Categorical features


,categorical_feature
